# Push clusters into the viewer, refine them there, pull them back

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/r_package/f_notebooks_tutorials/09_push_clusters_refine_and_pull.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Cluster with Seurat (`FindClusters`), show the clusters and a module
score in the viewer with `cellucid_push()`, fix the clusters by eye
(rename, merge), and bring the refined labels back into `meta.data`.

**Time:** 15 minutes. **You need:** the `Seurat` package. The same
tutorial in Python (Scanpy Leiden):
[Push clusters, refine, pull back](../../python_package/f_notebooks_tutorials/43_push_clusters_refine_and_pull.ipynb).

In [ ]:
for (package in c("remotes", "SeuratObject", "Seurat")) {
  if (!requireNamespace(package, quietly = TRUE)) install.packages(package)
}
if (!requireNamespace("cellucid", quietly = TRUE)) {
  if ("cellucid" %in% rownames(available.packages())) install.packages("cellucid") else remotes::install_github("theislab/cellucid-r")
}

## The data

SeuratObject's own example, `pbmc_small`: 80 blood cells, 230 genes, a
t-SNE and the clusters `RNA_snn_res.1`. Use your own Seurat or
SingleCellExperiment object in its place.

In [ ]:
library(cellucid)
library(SeuratObject)
data("pbmc_small")
seu <- pbmc_small
seu

## Cluster and score with Seurat

In [ ]:
library(Seurat)
seu <- NormalizeData(seu, verbose = FALSE)
seu <- FindVariableFeatures(seu, verbose = FALSE)
seu <- ScaleData(seu, verbose = FALSE)
seu <- RunPCA(seu, npcs = 10, verbose = FALSE)
seu <- FindNeighbors(seu, dims = 1:10, verbose = FALSE)
seu <- FindClusters(seu, resolution = 1, verbose = FALSE)
seu <- AddModuleScore(seu, features = list(c("MS4A1", "CD79A", "CD79B")), name = "B_score", ctrl = 5)
table(seu$seurat_clusters)

## Show and push

`cellucid_push(viewer, values, key)` sends a column at once: a factor
or character vector becomes a categorical field, numbers a continuous
one. A named vector (`seu$seurat_clusters` is named by cell) is
aligned by its names. Pushing the same key again replaces the field.
Run it once the viewer shows the cells: a column is checked against
the dataset the viewer opened, so before that it stops and says to
run it again.

In [ ]:
viewer <- cellucid_view(seu, dataset_id = "pbmc-small", dataset_name = "PBMC small")

In [ ]:
cellucid_push(viewer, seu$seurat_clusters, key = "seurat_clusters")
cellucid_push(viewer, seu$B_score1, key = "B_score")
cellucid_color_by(viewer, "seurat_clusters")

## Refine in the viewer

1. **Copy** the `seurat_clusters` field and **Rename** the copy
   `cell_type`.
2. In its legend, rename each cluster to a cell type (the pencil) and
   drag clusters of one type onto each other to merge them.
3. Colour by `B_score` to check the B cells.

Then run the next cell.

In [ ]:
pulled <- cellucid_pull(seu, viewer, return_summary = TRUE)
seu <- pulled$data
str(pulled$summary$skipped)
if ("cell_type" %in% colnames(seu[[]])) table(seu$seurat_clusters, seu$cell_type)

`cell_type` is your field, so it comes back. The two fields pushed from
R are skipped (`summary$skipped` says so): the object has them already.
`include_pushed = TRUE` brings them back too, for example after you
edited `seurat_clusters` itself rather than a copy.

Pushing again replaces the field: recluster at another resolution,
push under the same key, refine, pull again.

In [ ]:
seu <- FindClusters(seu, resolution = 0.5, verbose = FALSE)
cellucid_push(viewer, seu$seurat_clusters, key = "seurat_clusters")

In [ ]:
cellucid_stop(viewer)

## With a SingleCellExperiment

The same calls take a column of `colData`, named by cell:
`cellucid_push(viewer, setNames(sce$label, colnames(sce)), key = "label")`,
and `sce <- cellucid_pull(sce, viewer)` writes the refined field into
`colData(sce)`.

## Next

- [Gene sets round trip](11_gene_sets_round_trip.ipynb)
- Python: [Push clusters, refine, pull back](../../python_package/f_notebooks_tutorials/43_push_clusters_refine_and_pull.ipynb)